In [1]:
# check all the jsonl file if equal no of records in each file and if not then print the file name and number of records in that file check their "id" and for ragas_hallucination.jsonl file check "original_id" key

files = [
    "./ragas_hallucination.jsonl",
    "./baseline_score_filtered.jsonl",
    "./human_label.jsonl",
    "./selfcheck_dataset_filtered.jsonl",
    "./similarity_dataset_filtered.jsonl"
    ]



In [2]:
import json

human_labelled_data = "./human_label.jsonl"
baseline_file = "./baseline_score_filtered.jsonl"

# 1. Get all IDs from human-labeled data
with open(human_labelled_data, 'r', encoding='utf-8') as f:
    human_ids = {json.loads(line)['id'] for line in f if line.strip() and 'id' in json.loads(line)}

# 2. Get all IDs from baseline_score file
with open(baseline_file, 'r', encoding='utf-8') as f:
    baseline_ids = {json.loads(line)['id'] for line in f if line.strip() and 'id' in json.loads(line)}

# 3. Find the missing ID (the one in human data, but not in baseline)
missing_ids = human_ids - baseline_ids

print("Missing ID(s) in baseline_score.jsonl:", missing_ids)

Missing ID(s) in baseline_score.jsonl: {'c534ea0c-4253-47ba-bcb2-22e40834b88b'}


In [4]:
import json

files = [
    "./ragas_hallucination_results_filtered.jsonl",
    "./baseline_score_filtered.jsonl",
    "./human_label.jsonl",
    "./selfcheck_dataset_filtered.jsonl",
    "./similarity_dataset_filtered.jsonl"
]

file_stats = {}

for file_path in files:
    ids = set()
    count = 0
    id_key = "original_id" if "ragas_hallucination" in file_path else "id"
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                if not line.strip():
                    continue
                count += 1
                try:
                    data = json.loads(line)
                    if id_key in data:
                        ids.add(data[id_key])
                    else:
                        print(f"Warning: '{id_key}' missing in {file_path} (Line {line_num})")
                except json.JSONDecodeError:
                    print(f"Warning: Invalid JSON in {file_path} (Line {line_num})")
                    
        file_stats[file_path] = {'count': count, 'ids': ids}
        
    except FileNotFoundError:
        print(f"Error: File not found - {file_path}")

if len(file_stats) == len(files):
    # 1. Check if all counts are equal
    counts = [stat['count'] for stat in file_stats.values()]
    all_counts_match = len(set(counts)) == 1
    
    print("**RECORD COUNT CHECK**")
    if all_counts_match:
        print(f"✅ Success: All files have exactly {counts[0]} records.\n")
    else:
        print("❌ Mismatch: Record counts vary across files.")
        for file_path, stat in file_stats.items():
            print(f" - {file_path}: {stat['count']} records")
        print("\n")

    # 2. Check if all ID sets are identical
    id_sets = [stat['ids'] for stat in file_stats.values()]
    all_ids_match = all(s == id_sets[0] for s in id_sets)
    
    print("**ID IDENTICALITY CHECK**")
    if all_ids_match:
        print("✅ Success: All files contain the exact same set of IDs.")
    else:
        print("❌ Mismatch: The IDs are not identical across all files.\n")
        
        # Cross-reference to find missing or extra IDs
        master_id_set = set.union(*id_sets)
        for file_path, stat in file_stats.items():
            missing_ids = master_id_set - stat['ids']
            if missing_ids:
                sample = list(missing_ids)[:5]
                print(f" - {file_path} is missing {len(missing_ids)} IDs.")
                print(f"   Examples missing: {sample}")

**RECORD COUNT CHECK**
✅ Success: All files have exactly 300 records.

**ID IDENTICALITY CHECK**
✅ Success: All files contain the exact same set of IDs.
